In [ ]:
import numpy as np
import pandas as pd


In [ ]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
from surprise.model_selection import cross_validate

In [ ]:
path = "interactions_data/"

In [4]:
df_movies = pd.read_csv(path + 'movies_data.csv')
df_interactions = pd.read_csv(path + 'interactions_data.csv')

In [5]:
df_movies.head()

,id,title,description,genres,actors,directors
0,37702,Forbidden City Cop,An imperial agent gets ridiculed for his vario...,"Adventure, Action, Comedy","Stephen Chow, Carina Lau, Carman Lee Yeuk-Tung...",Vincent Kok Tak-Chiu
1,1290821,Shelter,A man living in self-imposed exile on a remote...,"Action, Thriller, Crime","Jason Statham, Bodhi Rae Breathnach, Bill Nigh...",Ric Roman Waugh
2,1171145,Crime 101,When an elusive thief whose high-stakes heists...,"Thriller, Crime","Chris Hemsworth, Mark Ruffalo, Halle Berry, Ba...",Bart Layton
3,1198994,Send Help,Two colleagues become stranded on a deserted i...,"Horror, Comedy, Thriller","Rachel McAdams, Dylan O'Brien, Edyll Ismail, D...",Sam Raimi
4,840464,Greenland 2: Migration,Having found the safety of the Greenland bunke...,"Adventure, Thriller, Science Fiction","Gerard Butler, Morena Baccarin, Roman Griffin ...",Ric Roman Waugh


In [6]:
df_interactions.head()

,user_id,movie_id,rating,max_progress_percent,favourite,in_watchlist,view_count,last_watched_at
0,10,1383731,NaN,19,0,0,1,2026-02-04 23:15:05.698446
1,10,915295,NaN,32,0,0,1,2025-11-04 23:15:05.699458
2,10,493899,NaN,85,0,0,5,2025-11-17 23:15:05.711448
3,10,28366,NaN,94,0,0,4,2025-12-16 23:15:05.726451
4,10,801,NaN,91,0,1,2,2025-12-09 23:15:05.738486


In [ ]:
def calculate_base_score(row, weights):
    v_progress = row['max_progress_percent'] / 100
    v_rating = row['rating'] / 5.0 if pd.notnull(row['rating']) else None
    v_fav = float(row['favourite'])
    v_watchlist = float(row['in_watchlist'])
    v_view_count = np.log(max(row['view_count'], 1)) / np.log(10)

    # Tính điểm theo từng thành phần hợp lệ
    numerator = (
        weights['progress'] * v_progress
        + weights['favourite'] * v_fav
        + weights['watchlist'] * v_watchlist
        + weights['view_count'] * v_view_count
    )
    denominator = (
        weights['progress']
        + weights['favourite']
        + weights['watchlist']
        + weights['view_count']
    )

    # Chỉ cộng rating khi có dữ liệu
    if v_rating is not None:
        numerator += weights['rating'] * v_rating
        denominator += weights['rating']

    return numerator / denominator * 10

In [ ]:
weights = {
    'progress': 0.4,
    'rating': 0.2,
    'favourite': 0.12,
    'watchlist': 0.08,
    'view_count': 0.2
}

df_interactions['base_score'] = df_interactions.apply(calculate_base_score, axis=1, weights=weights)

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [ ]:
df